# README
Base template for HACS202 honeypot project data analysis. Subject to change.

### **Instructions**
Please make a copy of this template for your honeypot project. Rename the file to follow this structure:

{TEAM NUMBER}\_{TEAM NAME}\_HACS202\_Data

Additionally, please keep this document as organized as you can. Consider commenting your code and/or providing some sort of documentation. This will make it easier for you, your teammates, and course staff to understand what has been written in the long run!

***
# Team Information
Team Name: HoneyPot Chickens

Team Members: Zayd Mahfuz, Santosh Sureshkumar, Navtej Soma, FNU Mahek

***
# Set Up
Importing libraries. Please run this first.

In [ ]:
# importing libraries
!pip install scikit-posthocs

In [ ]:
# more library imports
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm
import scikit_posthocs as skph
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import MultiComparison
from sklearn.datasets import load_iris

***
# File Uploads
Upload your data here by running the cell below!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# allows your to upload data to colab
from google.colab import files
# uploads = files.upload()

file_path = '/content/drive/MyDrive/HACS202 - Honeypot Project/2_all_durations_raw_UPDATED.csv'

df = pd.read_csv(file_path)
df["first_dt"] = pd.to_datetime("2024 " + df["first_timestamp"])
df["last_dt"]  = pd.to_datetime("2024 " + df["last_timestamp"])

# Duration in seconds (float, can include milliseconds)
df["duration_milliseconds"] = (df["last_dt"] - df["first_dt"]).dt.total_seconds() * 1000



In [ ]:
from google.colab import files

# Paths for the new CSVs
file_path_cmds_all    = "/content/drive/MyDrive/HACS202 - Honeypot Project/commands_all.csv"
file_path_cmds_unique = "/content/drive/MyDrive/HACS202 - Honeypot Project/commands_unique_ips.csv"

# Read the CSVs
df_commands_all    = pd.read_csv(file_path_cmds_all)
df_commands_unique = pd.read_csv(file_path_cmds_unique)


***
# Violin Plots

Please create the required plots listed below. You should be creating violin plots with boxes. Please reference the examples in the modules on ELMS if you are unsure of what the plots are supposed to look like! You are welcome to add additional markdown/code blocks!

### **Required Plots**
You will need to make a plot for the following:
- Time spent per configuration
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration
- Number of commands per configuration
- Number of commands (unique IPs) per configuration

_You are welcome to make additional plots as you see fit. Incorrect/misleading plots may result in a penalty._

### **Documentation can be easily found online for additional functions!**
Suggestions:
- Start a new code block for each plot
- Use `violin()` from `plotly.express` to create your violin plots
  - Example: `px.violin(...)`
- Assign your violin plot a descriptive name
  - Example: `fig_ex = px.violin(...)`
  - Here, "fig_ex" is the name of the violin plot
- Ensure that you are creating violin plots with boxes
  - You can use `box=True` as one of the arguments when making violin plots
- You can use `show()` to display your violin plots
  - Example: `fig.show()`

In [ ]:
import plotly.express as px
import pandas as pd

# -------------------------
# 0. REMOVE NEGATIVE VALUES
# -------------------------

df = df[df["duration_milliseconds"] >= 0].copy()

df_commands_all = df_commands_all[df_commands_all["num_commands"] >= 0].copy()
df_commands_unique = df_commands_unique[df_commands_unique["num_commands"] >= 0].copy()


# ---------------------------------------------------------
# 1. First Graph — All attack durations per configuration
# ---------------------------------------------------------
fig_time_per_config = px.violin(
    df,
    x="honeypot",
    y="duration_milliseconds",
    box=True,
    points="all",
    title="Time Spent per Configuration (milliseconds)"
)
fig_time_per_config.update_yaxes(tickformat=".0f")
fig_time_per_config.show()


# --------------------------------------------------------------------
# 2. Second Graph — All durations excluding timeouts (no negative, no 0)
# --------------------------------------------------------------------
df_no_timeouts = df[df["duration_milliseconds"] > 0].copy()

fig_time_no_timeouts = px.violin(
    df_no_timeouts,
    x="honeypot",
    y="duration_milliseconds",
    box=True,
    points="all",
    title="Time Spent (Excluding Timeouts) per Configuration (milliseconds)"
)
fig_time_no_timeouts.update_yaxes(tickformat=".0f")
fig_time_no_timeouts.show()


# ---------------------------------------------------------------------
# 3. Third Graph — Unique IP total durations (negatives already removed)
# ---------------------------------------------------------------------
df_ip_time = (
    df.groupby(["honeypot", "source_ip"], as_index=False)["duration_milliseconds"]
      .sum()
      .rename(columns={"duration_milliseconds": "total_duration_milliseconds"})
)

fig_time_unique_ips = px.violin(
    df_ip_time,
    x="honeypot",
    y="total_duration_milliseconds",
    box=True,
    points="all",
    title="Time Spent (Unique IPs) per Configuration (milliseconds)"
)
fig_time_unique_ips.update_yaxes(tickformat=".0f")
fig_time_unique_ips.show()


# ---------------------------------------------------------------------------
# 4. Fourth Graph — Unique IP durations excluding timeouts (no neg, no zeros)
# ---------------------------------------------------------------------------
df_ip_time_no_timeouts = (
    df_no_timeouts.groupby(["honeypot", "source_ip"], as_index=False)["duration_milliseconds"]
                  .sum()
                  .rename(columns={"duration_milliseconds": "total_duration_milliseconds"})
)

fig_time_unique_ips_no_timeouts = px.violin(
    df_ip_time_no_timeouts,
    x="honeypot",
    y="total_duration_milliseconds",
    box=True,
    points="all",
    title="Time Spent (Unique IPs, Excluding Timeouts) per Configuration (milliseconds)"
)
fig_time_unique_ips_no_timeouts.update_yaxes(tickformat=".0f")
fig_time_unique_ips_no_timeouts.show()


# ------------------------------------------
# 5. Fifth Graph — Number of Commands per HP configuration
# ------------------------------------------
fig_cmds_all = px.violin(
    df_commands_all,
    x="honeypot",
    y="num_commands",
    box=True,
    points="all",
    title="Number of Commands per Configuration"
)
fig_cmds_all.show()


# --------------------------------------------------------
# 6. Sixth Graph — Number of Commands from unique IPs per config
# --------------------------------------------------------
fig_cmds_unique = px.violin(
    df_commands_unique,
    x="honeypot",
    y="num_commands",
    box=True,
    points="all",
    title="Number of Commands (Unique IPs) per Configuration"
)
fig_cmds_unique.show()


***
# Normality Tests

You are welcome to add additional markdown/code blocks!

### **Required Analyses**
You will need to conduct normality tests on the following:
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration

_You are welcome to analyze additional data sets as you see fit. Incorrect/misleading analyses may result in a penalty._

### **Please use the following modules to perform the normality tests:**
Kolmogorov-Smirnov (KS) - use `kstest()` from `scipy.stats`
- If you did not modify the imports, the command should be `sp.stats.ktest(...)`

### **Remember that the documentation for all of these libraries and modules can be easily found online!**
Suggestions:
- Each test you conduct/dataset you investigate should be in its own code block
- Group tests together based on type (i.e. all ANOVA tests should be together, etc.)

In [ ]:
import scipy as sp

# Time spent (excluding timeouts) per configuration that is df_no_timeouts

results_time_no_timeouts = {} # maps honeypot to the dict containing ks stat and p value for that honeypot

# df_no_timeouts

for config in df_no_timeouts["honeypot"].unique():
    data = df_no_timeouts[df_no_timeouts["honeypot"] == config]["duration_milliseconds"]
    ks_stat, p_value = sp.stats.kstest(data, "norm")
    results_time_no_timeouts[config] = {"KS Statistic": ks_stat, "p-value": p_value}

results_time_no_timeouts

{'container-control': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment1': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment2': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment3': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)}}

### Observations:

The violin plot for attack durations excluding timeouts shows that all four honeypot configurations still have a heavily right-skewed distribution, with most attacks clustered very close to zero seconds and only a small number of longer interaction sessions stretching upward. While there are a few long-duration points for each configuration, the density clusters are near the bottom of the plot, and the shapes of the violins are similar across control and the three treatments.

This matches the KS test results. Every configuration returned a KS statistic of 1.0 and a p-value of 0.0, which means that none of these distributions resemble a normal distribution. The data is super skewed, with most attacks happening really fast and only a few lasting longer. Both the plot and the KS results show the same thing that the attack times are not normal at all and they look basically the same across every configuration.

In [ ]:
# Time spent (unique IPs, excluding timeouts) per configuration that is df_no_timeouts

results_unique_ips = {}

for config in df_ip_time_no_timeouts["honeypot"].unique():
    data = df_ip_time_no_timeouts[df_ip_time_no_timeouts["honeypot"] == config]["total_duration_milliseconds"]
    stat, p = sp.stats.kstest(data, "norm")
    results_unique_ips[config] = {"KS Statistic": stat, "p-value": p}

results_unique_ips

{'container-control': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment1': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment2': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)},
 'container-treatment3': {'KS Statistic': np.float64(1.0),
  'p-value': np.float64(0.0)}}

### Observations:

The KS results for the unique IP durations without timeouts look exactly like the earlier ones. Every configuration has a KS value of 1.0 and a p-value of 0.0, which means the data is still nowhere close to normal. Even after grouping by unique IPs and removing the timeout entries, the distributions remain extremely skewed with only a few long sessions and most of the values staying near the lower end.

This can also be observed from the violin plot. Almost all of the density for each configuration is packed near zero, while only a small number of IPs stretch the tail upward. Both the plot and the KS results show the same thing that the attack times are not normal at all and they look basically the same across every configuration.

***
# Statistical Tests
You will need to conduct some statistical tests on your data. Consider refering to class slides and the statistics homework colab if you need a refresher on any of the tests. You are welcome to add additional markdown/code blocks!

### **Required Analyses**
You will need to conduct tests on the following:
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration
- Number of commands per configuration
- Number of commands (unique IPs) per configuration

_You are welcome to analyze additional data sets as you see fit. Incorrect/misleading analyses may result in a penalty._

### **Please use the following modules to perform the statistical tests:**

ANOVA - use `anova_lm()` from `statsmodels.api.stats` (fancy ANOVA)
  - If you did not modify the imports, the command should be `sm.stats.anova_lm(...)`

TUKEY HSD - use `pairwise_tukeyhsd()`
  - You only need to run a Tukey HSD test if ANOVA is significant (p-value < 0.1)
  - Consider using an if statement!

KRUSKAL-WALLIS - use `kruskal()` from `scipy.stats`
  - If you did not modify the imports, the command should be `sp.stats.kruskal(...)`

DUNN - use `posthoc_dunn()` from `scikit_posthocs`
  - You only need to run a Dunn test if Kruskal-Wallis is significant (p-value < 0.1)
  - If you did not modify the imports, the command should be `spkh.posthoc_dunn(...)`

CHI SQUARED - use `chisquare()` from `scipy.stats`
- If you did not modify the imports, the command should be `sp.stats.chisquare(...)`
### **Remember that the documentation for all of these libraries and modules can be easily found online!**
Suggestions:
- Each test you conduct should be in its own code block
- Group tests together based on type (i.e. all ANOVA tests should be together, etc.)

In [ ]:
# ANOVA Test for second graph: all attack durations per HP configuration with no timeouts
print("\n" + "="*80)
print("TEST 1: ANOVA - Time Spent per Configuration (Excluding Timeouts, ms)")
print("="*80)

# ANOVA on durations excluding timeouts
model_time = ols('duration_milliseconds ~ C(honeypot)', data=df_no_timeouts).fit()
anova_time = sm.stats.anova_lm(model_time, typ=2)
print(anova_time)

anova_time_p = anova_time['PR(>F)'][0]
print(f"\nP-value: {anova_time_p:.4f}")

if anova_time_p < 0.1:
    print(f"✓ ANOVA SIGNIFICANT (p={anova_time_p:.4f} < 0.1)")
else:
    print(f"✗ ANOVA NOT SIGNIFICANT (p={anova_time_p:.4f} ≥ 0.1)")


TEST 1: ANOVA - Time Spent per Configuration (Excluding Timeouts, ms)
                   sum_sq      df          F        PR(>F)
C(honeypot)  2.785789e+17     3.0  17.555483  2.382796e-11
Residual     3.571466e+19  6752.0        NaN           NaN

P-value: 0.0000
✓ ANOVA SIGNIFICANT (p=0.0000 < 0.1)


/tmp/ipython-input-14064394.py:11: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [ ]:
# if the pvalue<0.1 for anova, apply a Tukey HSD test
# TUKEY HSD for time spent per configuration (excluding timeouts)
if anova_time_p < 0.1:
    print("\n" + "="*80)
    print("TEST : TUKEY HSD POST-HOC (Time Durations, No Timeouts)")
    print("="*80)

    tukey_time = pairwise_tukeyhsd(
        endog=df_no_timeouts['duration_milliseconds'],
        groups=df_no_timeouts['honeypot'],
        alpha=0.1
    )
    print(tukey_time)

else:
    print("\nSkipping Tukey HSD for durations (ANOVA not significant)")


TEST : TUKEY HSD POST-HOC (Time Durations, No Timeouts)
                     Multiple Comparison of Means - Tukey HSD, FWER=0.10                     
       group1               group2          meandiff   p-adj     lower        upper    reject
---------------------------------------------------------------------------------------------
   container-control container-treatment1 -226847.6139 0.0004 -356480.1203 -97215.1074   True
   container-control container-treatment2   64594.7261 0.6914  -70350.5582 199540.0105  False
   container-control container-treatment3   23531.7009 0.9791 -112938.9549 160002.3566  False
container-treatment1 container-treatment2    291442.34    0.0  161987.6912 420896.9888   True
container-treatment1 container-treatment3  250379.3147 0.0001  119335.3671 381423.2623   True
container-treatment2 container-treatment3  -41063.0253 0.9008 -177364.7466  95238.6961  False
---------------------------------------------------------------------------------------------


In [ ]:
print("\n" + "="*80)
print("TEST : KRUSKAL-WALLIS - Time Spent per Configuration (Time Durations, No Timeouts)")
print("="*80)

# Prepare groups for Kruskal-Wallis
groups_time = [
    df_no_timeouts[df_no_timeouts["honeypot"] == hp]["duration_milliseconds"].values
    for hp in df_no_timeouts["honeypot"].unique()
]

# Kruskal-Wallis test
kw_time_stat, kw_time_p = sp.stats.kruskal(*groups_time)
print(f"H-statistic: {kw_time_stat:.4f}")
print(f"P-value: {kw_time_p:.4f}")

if kw_time_p < 0.1:
    print(f"\n✓ KRUSKAL-WALLIS SIGNIFICANT (p={kw_time_p:.4f} < 0.1)")
    print("→ Significant difference in attack duration distributions across honeypots")
else:
    print(f"\n✗ KRUSKAL-WALLIS NOT SIGNIFICANT (p={kw_time_p:.4f} ≥ 0.1)")
    print("→ No significant difference in attack duration distributions across honeypots")



TEST : KRUSKAL-WALLIS - Time Spent per Configuration (Time Durations, No Timeouts)
H-statistic: 53.4336
P-value: 0.0000

✓ KRUSKAL-WALLIS SIGNIFICANT (p=0.0000 < 0.1)
→ Significant difference in attack duration distributions across honeypots


In [ ]:
# DUNN post-hoc for all commands
if kw_time_p < 0.1:
    print("\n" + "="*80)
    print("TEST: DUNN POST-HOC (Time Durations, No Timeouts)")
    print("="*80)

    dunn_time = skph.posthoc_dunn(
        df_no_timeouts,
        val_col='duration_milliseconds',
        group_col='honeypot',
        p_adjust='bonferroni'
    )

    print(dunn_time)
    print("\nInterpretation: Values < 0.1 indicate significant pairwise differences")

else:
    print("\nSkipping Dunn test for durations (Kruskal-Wallis not significant)")


TEST: DUNN POST-HOC (Time Durations, No Timeouts)
                      container-control  container-treatment1  \
container-control          1.000000e+00          4.370392e-07   
container-treatment1       4.370392e-07          1.000000e+00   
container-treatment2       1.000000e+00          2.829564e-10   
container-treatment3       1.000000e+00          1.296699e-06   

                      container-treatment2  container-treatment3  
container-control             1.000000e+00              1.000000  
container-treatment1          2.829564e-10              0.000001  
container-treatment2          1.000000e+00              1.000000  
container-treatment3          1.000000e+00              1.000000  

Interpretation: Values < 0.1 indicate significant pairwise differences


In [ ]:
print("="*80)
print("COMMANDS_ALL.CSV")
print("="*80)
print(f"Total rows: {len(df_commands_all)}")
print(f"\nFirst few rows:")
print(df_commands_all.head(10))
print(f"\nSummary by honeypot:")
print(df_commands_all.groupby('honeypot')['num_commands'].describe())

print("\n" + "="*80)
print("COMMANDS_UNIQUE_IPS.CSV")
print("="*80)
print(f"Total rows: {len(df_commands_unique)}")
print(f"\nFirst few rows:")
print(df_commands_unique.head(10))
print(f"\nSummary by honeypot:")
print(df_commands_unique.groupby('honeypot')['num_commands'].describe())

COMMANDS_ALL.CSV
Total rows: 256

First few rows:
            honeypot  num_commands
0  container-control            14
1  container-control            19
2  container-control             9
3  container-control            17
4  container-control            17
5  container-control            23
6  container-control            20
7  container-control            14
8  container-control            11
9  container-control            11

Summary by honeypot:
                      count       mean        std   min   25%   50%    75%  \
honeypot                                                                     
container-control      88.0  16.068182   8.016930   1.0  11.0  16.5  21.00   
container-treatment1   53.0  32.735849  11.166831   9.0  26.0  34.0  40.00   
container-treatment2   60.0  54.883333  21.044015  11.0  42.0  60.0  68.25   
container-treatment3   55.0  76.727273  29.012420  12.0  55.0  71.0  97.00   

                        max  
honeypot                     
container-cont

In [ ]:
# CHI-SQUARED TEST: Total Commands per Configuration
print("\n" + "="*80)
print("TEST 1: CHI-SQUARED - Total Commands per Configuration")
print("="*80)

# Sum total commands per honeypot
total_commands = df_commands_all.groupby('honeypot')['num_commands'].sum()
print("\nTotal commands by honeypot:")
print(total_commands)

# Chi-squared test
chi2_stat, chi2_p = sp.stats.chisquare(total_commands.values)
print(f"\nChi-squared statistic: {chi2_stat:.4f}")
print(f"P-value: {chi2_p:.4f}")

if chi2_p < 0.1:
    print(f"\n✓ SIGNIFICANT (p={chi2_p:.4f} < 0.1)")
    print("→ Significant difference in total command counts between configurations")
else:
    print(f"\n✗ NOT SIGNIFICANT (p={chi2_p:.4f} ≥ 0.1)")
    print("→ No significant difference in total command counts")


TEST 1: CHI-SQUARED - Total Commands per Configuration

Total commands by honeypot:
honeypot
container-control       1414
container-treatment1    1735
container-treatment2    3293
container-treatment3    4220
Name: num_commands, dtype: int64

Chi-squared statistic: 1966.7263
P-value: 0.0000

✓ SIGNIFICANT (p=0.0000 < 0.1)
→ Significant difference in total command counts between configurations


In [ ]:
# ANOVA TEST: Commands per Unique IP
print("\n" + "="*80)
print("TEST 2: ANOVA - Number of Commands per Configuration (All Data)")
print("="*80)

# ANOVA on all commands
model_all = ols('num_commands ~ C(honeypot)', data=df_commands_all).fit()
anova_all = sm.stats.anova_lm(model_all, typ=2)
print(anova_all)

anova_all_p = anova_all['PR(>F)'][0]
print(f"\nP-value: {anova_all_p:.4f}")

if anova_all_p < 0.1:
    print(f"✓ ANOVA SIGNIFICANT (p={anova_all_p:.4f} < 0.1)")
else:
    print(f"✗ ANOVA NOT SIGNIFICANT (p={anova_all_p:.4f} ≥ 0.1)")


TEST 2: ANOVA - Number of Commands per Configuration (All Data)
                    sum_sq     df           F        PR(>F)
C(honeypot)  139981.374155    3.0  140.555333  1.569115e-53
Residual      83656.985220  252.0         NaN           NaN

P-value: 0.0000
✓ ANOVA SIGNIFICANT (p=0.0000 < 0.1)


/tmp/ipython-input-3476477936.py:11: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [ ]:
# TUKEY HSD for all commands
if anova_all_p < 0.1:
    print("\n" + "="*80)
    print("TEST 3: TUKEY HSD POST-HOC (All Commands)")
    print("="*80)

    tukey_all = pairwise_tukeyhsd(
        endog=df_commands_all['num_commands'],
        groups=df_commands_all['honeypot'],
        alpha=0.1
    )
    print(tukey_all)
else:
    print("\nSkipping Tukey HSD for all commands (ANOVA not significant)")


TEST 3: TUKEY HSD POST-HOC (All Commands)
              Multiple Comparison of Means - Tukey HSD, FWER=0.10              
       group1               group2        meandiff p-adj  lower   upper  reject
-------------------------------------------------------------------------------
   container-control container-treatment1  16.6677   0.0  9.3711 23.9642   True
   container-control container-treatment2  38.8152   0.0 31.7892 45.8411   True
   container-control container-treatment3  60.6591   0.0 53.4458 67.8724   True
container-treatment1 container-treatment2  22.1475   0.0 14.2368 30.0582   True
container-treatment1 container-treatment3  43.9914   0.0 35.9138  52.069   True
container-treatment2 container-treatment3  21.8439   0.0   14.01 29.6779   True
-------------------------------------------------------------------------------


In [ ]:
print("\n" + "="*80)
print("TEST 4: KRUSKAL-WALLIS - Commands per Configuration (All Data)")
print("="*80)

# Prepare groups for Kruskal-Wallis
groups_all = [
    df_commands_all[df_commands_all["honeypot"] == hp]["num_commands"].values
    for hp in df_commands_all["honeypot"].unique()
]

# Kruskal-Wallis test
kw_all_stat, kw_all_p = sp.stats.kruskal(*groups_all)
print(f"H-statistic: {kw_all_stat:.4f}")
print(f"P-value: {kw_all_p:.4f}")

if kw_all_p < 0.1:
    print(f"\n✓ KRUSKAL-WALLIS SIGNIFICANT (p={kw_all_p:.4f} < 0.1)")
    print("→ Significant difference in command distributions")
else:
    print(f"\n✗ KRUSKAL-WALLIS NOT SIGNIFICANT (p={kw_all_p:.4f} ≥ 0.1)")
    print("→ No significant difference in command distributions")


TEST 4: KRUSKAL-WALLIS - Commands per Configuration (All Data)
H-statistic: 172.4929
P-value: 0.0000

✓ KRUSKAL-WALLIS SIGNIFICANT (p=0.0000 < 0.1)
→ Significant difference in command distributions


In [ ]:
# DUNN post-hoc for all commands
if kw_all_p < 0.1:
    print("\n" + "="*80)
    print("TEST 5: DUNN POST-HOC (All Commands)")
    print("="*80)

    dunn_all = skph.posthoc_dunn(
        df_commands_all,
        val_col='num_commands',
        group_col='honeypot',
        p_adjust='bonferroni'
    )
    print(dunn_all)
    print("\nInterpretation: Values < 0.1 indicate significant pairwise differences")
else:
    print("\nSkipping Dunn test for all commands (Kruskal-Wallis not significant)")


TEST 5: DUNN POST-HOC (All Commands)
                      container-control  container-treatment1  \
container-control          1.000000e+00          1.502889e-06   
container-treatment1       1.502889e-06          1.000000e+00   
container-treatment2       4.236250e-21          9.384294e-04   
container-treatment3       2.804387e-32          9.416999e-09   

                      container-treatment2  container-treatment3  
container-control             4.236250e-21          2.804387e-32  
container-treatment1          9.384294e-04          9.416999e-09  
container-treatment2          1.000000e+00          9.647896e-02  
container-treatment3          9.647896e-02          1.000000e+00  

Interpretation: Values < 0.1 indicate significant pairwise differences


In [ ]:
# Using your existing df from 2_all_durations_raw.csv
# It already has duration_seconds calculated

# Define time limit (180 minutes = 10,800 seconds)
TIME_LIMIT_MILLISECONDS = 180 * 60 * 1000 # 10,800 seconds

print(f"Total rows in original data: {len(df)}")
print(f"Rows with duration >= {TIME_LIMIT_MILLISECONDS} seconds (timeouts): {len(df[df['duration_milliseconds'] >= TIME_LIMIT_MILLISECONDS])}")

# Create df_no_timeouts (excluding timeouts)
# Note: You already have this, but let's make sure it excludes the time limit
df_no_timeouts = df[df["duration_milliseconds"] < TIME_LIMIT_MILLISECONDS].copy()
print(f"Rows after excluding timeouts: {len(df_no_timeouts)}")

# Create unique IPs dataset (excluding timeouts)
df_ip_time_no_timeouts = (
    df_no_timeouts.groupby(["honeypot", "source_ip"], as_index=False)["duration_milliseconds"]
                  .sum()
                  .rename(columns={"duration_milliseconds": "total_duration_milliseconds"})
)
print(f"Unique IPs (no timeouts): {len(df_ip_time_no_timeouts)}")

Total rows in original data: 9855
Rows with duration >= 10800000 seconds (timeouts): 846
Rows after excluding timeouts: 9009
Unique IPs (no timeouts): 9009


In [ ]:
print("\n" + "="*80)
print("ANOVA TEST: Time Spent (Unique IPs, Excluding Timeouts)")
print("="*80)

# ANOVA
model_unique_time = ols('total_duration_milliseconds ~ C(honeypot)', data=df_ip_time_no_timeouts).fit()
anova_unique_time = sm.stats.anova_lm(model_unique_time, typ=2)
print(anova_unique_time)

anova_unique_time_p = anova_unique_time['PR(>F)'][0]
print(f"\nP-value: {anova_unique_time_p:.4f}")


if anova_unique_time_p < 0.1:
    print(f"✓ ANOVA SIGNIFICANT (p={anova_unique_time_p:.4f} < 0.1)")
else:
    print(f"✗ ANOVA NOT SIGNIFICANT (p={anova_unique_time_p:.4f} ≥ 0.1)")


ANOVA TEST: Time Spent (Unique IPs, Excluding Timeouts)
                   sum_sq      df         F        PR(>F)
C(honeypot)  1.255886e+14     3.0  11.10944  2.826666e-07
Residual     3.393288e+16  9005.0       NaN           NaN

P-value: 0.0000
✓ ANOVA SIGNIFICANT (p=0.0000 < 0.1)


/tmp/ipython-input-1961012805.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [ ]:
# TUKEY HSD (if ANOVA significant)
if anova_unique_time_p < 0.1:
    print("\n" + "="*80)
    print("TUKEY HSD POST-HOC: Time Spent (Unique IPs, Excluding Timeouts)")
    print("="*80)

    tukey_unique_time = pairwise_tukeyhsd(
        endog=df_ip_time_no_timeouts['total_duration_milliseconds'],
        groups=df_ip_time_no_timeouts['honeypot'],
        alpha=0.1
    )
    print(tukey_unique_time)
else:
    print("\nSkipping Tukey HSD (ANOVA not significant)")


TUKEY HSD POST-HOC: Time Spent (Unique IPs, Excluding Timeouts)
                     Multiple Comparison of Means - Tukey HSD, FWER=0.10                     
       group1               group2          meandiff   p-adj     lower        upper    reject
---------------------------------------------------------------------------------------------
   container-control container-treatment1 -226847.6139 0.0004 -356480.1203 -97215.1074   True
   container-control container-treatment2   64594.7261 0.6914  -70350.5582 199540.0105  False
   container-control container-treatment3   23531.7009 0.9791 -112938.9549 160002.3566  False
container-treatment1 container-treatment2    291442.34    0.0  161987.6912 420896.9888   True
container-treatment1 container-treatment3  250379.3147 0.0001  119335.3671 381423.2623   True
container-treatment2 container-treatment3  -41063.0253 0.9008 -177364.7466  95238.6961  False
-----------------------------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("KRUSKAL-WALLIS TEST: Time Spent (Unique IPs, Excluding Timeouts)")
print("="*80)

# Prepare groups
groups_unique_time = [
    df_ip_time_no_timeouts[df_ip_time_no_timeouts["honeypot"] == hp]["total_duration_milliseconds"].values
    for hp in df_ip_time_no_timeouts["honeypot"].unique()
]

# Kruskal-Wallis test
kw_unique_time_stat, kw_unique_time_p = sp.stats.kruskal(*groups_unique_time)
print(f"H-statistic: {kw_unique_time_stat:.4f}")
print(f"P-value: {kw_unique_time_p:.4f}")

if kw_unique_time_p < 0.1:
    print(f"\n✓ KRUSKAL-WALLIS SIGNIFICANT (p={kw_unique_time_p:.4f} < 0.1)")
else:
    print(f"\n✗ KRUSKAL-WALLIS NOT SIGNIFICANT (p={kw_unique_time_p:.4f} ≥ 0.1)")


KRUSKAL-WALLIS TEST: Time Spent (Unique IPs, Excluding Timeouts)
H-statistic: 53.4336
P-value: 0.0000

✓ KRUSKAL-WALLIS SIGNIFICANT (p=0.0000 < 0.1)


In [ ]:
# DUNN POST-HOC (if Kruskal-Wallis significant)
if kw_unique_time_p < 0.1:
    print("\n" + "="*80)
    print("DUNN POST-HOC: Time Spent (Unique IPs, Excluding Timeouts)")
    print("="*80)

    dunn_unique_time = skph.posthoc_dunn(
        df_ip_time_no_timeouts,
        val_col='total_duration_milliseconds',
        group_col='honeypot',
        p_adjust='bonferroni'
    )
    print(dunn_unique_time)
    print("\nInterpretation: Values < 0.1 indicate significant pairwise differences")
else:
    print("\nSkipping Dunn test (Kruskal-Wallis not significant)")


DUNN POST-HOC: Time Spent (Unique IPs, Excluding Timeouts)
                      container-control  container-treatment1  \
container-control          1.000000e+00          4.370392e-07   
container-treatment1       4.370392e-07          1.000000e+00   
container-treatment2       1.000000e+00          2.829564e-10   
container-treatment3       1.000000e+00          1.296699e-06   

                      container-treatment2  container-treatment3  
container-control             1.000000e+00              1.000000  
container-treatment1          2.829564e-10              0.000001  
container-treatment2          1.000000e+00              1.000000  
container-treatment3          1.000000e+00              1.000000  

Interpretation: Values < 0.1 indicate significant pairwise differences


In [ ]:
print("\n" + "="*80)
print("ANOVA: Number of Commands per Configuration (Unique IPs)")
print("="*80)

# ANOVA on num_commands for unique IPs
model_cmds_unique = ols('num_commands ~ C(honeypot)', data=df_commands_unique).fit()
anova_cmds_unique = sm.stats.anova_lm(model_cmds_unique, typ=2)
print(anova_cmds_unique)

# Get p-value for honeypot factor
anova_cmds_unique_p = anova_cmds_unique["PR(>F)"].iloc[0]
print(f"\nP-value: {anova_cmds_unique_p:.4f}")

if anova_cmds_unique_p < 0.1:
    print(f"✓ ANOVA SIGNIFICANT (p={anova_cmds_unique_p:.4f} < 0.1)")
    print("\n" + "="*80)
    print("TUKEY HSD POST-HOC: Number of Commands (Unique IPs)")
    print("="*80)

    tukey_cmds_unique = pairwise_tukeyhsd(
        endog=df_commands_unique['num_commands'],
        groups=df_commands_unique['honeypot'],
        alpha=0.1
    )
    print(tukey_cmds_unique)
else:
    print(f"✗ ANOVA NOT SIGNIFICANT (p={anova_cmds_unique_p:.4f} ≥ 0.1)")
    print("Skipping Tukey HSD (ANOVA not significant)")



ANOVA: Number of Commands per Configuration (Unique IPs)
                    sum_sq     df           F        PR(>F)
C(honeypot)  139981.374155    3.0  140.555333  1.569115e-53
Residual      83656.985220  252.0         NaN           NaN

P-value: 0.0000
✓ ANOVA SIGNIFICANT (p=0.0000 < 0.1)

TUKEY HSD POST-HOC: Number of Commands (Unique IPs)
              Multiple Comparison of Means - Tukey HSD, FWER=0.10              
       group1               group2        meandiff p-adj  lower   upper  reject
-------------------------------------------------------------------------------
   container-control container-treatment1  16.6677   0.0  9.3711 23.9642   True
   container-control container-treatment2  38.8152   0.0 31.7892 45.8411   True
   container-control container-treatment3  60.6591   0.0 53.4458 67.8724   True
container-treatment1 container-treatment2  22.1475   0.0 14.2368 30.0582   True
container-treatment1 container-treatment3  43.9914   0.0 35.9138  52.069   True
container-treat

In [ ]:
print("\n" + "="*80)
print("KRUSKAL-WALLIS: Number of Commands per Configuration (Unique IPs)")
print("="*80)

# Prepare groups by honeypot
groups_cmds_unique = [
    df_commands_unique[df_commands_unique["honeypot"] == hp]["num_commands"].values
    for hp in df_commands_unique["honeypot"].unique()
]

kw_cmds_unique_stat, kw_cmds_unique_p = sp.stats.kruskal(*groups_cmds_unique)
print(f"H-statistic: {kw_cmds_unique_stat:.4f}")
print(f"P-value: {kw_cmds_unique_p:.4f}")

if kw_cmds_unique_p < 0.1:
    print(f"\n✓ KRUSKAL-WALLIS SIGNIFICANT (p={kw_cmds_unique_p:.4f} < 0.1)")
    print("→ Significant difference in command distributions (unique IPs) across honeypots")

    print("\n" + "="*80)
    print("DUNN POST-HOC: Number of Commands (Unique IPs)")
    print("="*80)

    dunn_cmds_unique = skph.posthoc_dunn(
        df_commands_unique,
        val_col='num_commands',
        group_col='honeypot',
        p_adjust='bonferroni'
    )
    print(dunn_cmds_unique)
    print("\nInterpretation: Values < 0.1 indicate significant pairwise differences")

else:
    print(f"\n✗ KRUSKAL-WALLIS NOT SIGNIFICANT (p={kw_cmds_unique_p:.4f} ≥ 0.1)")
    print("Skipping Dunn test (Kruskal-Wallis not significant)")



KRUSKAL-WALLIS: Number of Commands per Configuration (Unique IPs)
H-statistic: 172.4929
P-value: 0.0000

✓ KRUSKAL-WALLIS SIGNIFICANT (p=0.0000 < 0.1)
→ Significant difference in command distributions (unique IPs) across honeypots

DUNN POST-HOC: Number of Commands (Unique IPs)
                      container-control  container-treatment1  \
container-control          1.000000e+00          1.502889e-06   
container-treatment1       1.502889e-06          1.000000e+00   
container-treatment2       4.236250e-21          9.384294e-04   
container-treatment3       2.804387e-32          9.416999e-09   

                      container-treatment2  container-treatment3  
container-control             4.236250e-21          2.804387e-32  
container-treatment1          9.384294e-04          9.416999e-09  
container-treatment2          1.000000e+00          9.647896e-02  
container-treatment3          9.647896e-02          1.000000e+00  

Interpretation: Values < 0.1 indicate significant pairwise

In [ ]:
print("\n" + "="*80)
print("ANOVA: Number of Commands During Each Attack per HP Configuration")
print("="*80)

# ANOVA model: num_commands explained by honeypot configuration
model_attack_cmds = ols('num_commands ~ C(honeypot)', data=df_commands_all).fit()
anova_attack_cmds = sm.stats.anova_lm(model_attack_cmds, typ=2)
print(anova_attack_cmds)

# Extract p-value for the honeypot factor
anova_attack_cmds_p = anova_attack_cmds["PR(>F)"].iloc[0]
print(f"\nP-value: {anova_attack_cmds_p:.4f}")

if anova_attack_cmds_p < 0.1:
    print(f"✓ ANOVA SIGNIFICANT (p={anova_attack_cmds_p:.4f} < 0.1)")
    print("\n" + "="*80)
    print("TUKEY HSD POST-HOC: Number of Commands During Each Attack")
    print("="*80)

    tukey_attack_cmds = pairwise_tukeyhsd(
        endog=df_commands_all["num_commands"],
        groups=df_commands_all["honeypot"],
        alpha=0.1
    )
    print(tukey_attack_cmds)

else:
    print(f"✗ ANOVA NOT SIGNIFICANT (p={anova_attack_cmds_p:.4f} ≥ 0.1)")
    print("Skipping Tukey HSD (ANOVA not significant)")



ANOVA: Number of Commands During Each Attack per HP Configuration
                    sum_sq     df           F        PR(>F)
C(honeypot)  139981.374155    3.0  140.555333  1.569115e-53
Residual      83656.985220  252.0         NaN           NaN

P-value: 0.0000
✓ ANOVA SIGNIFICANT (p=0.0000 < 0.1)

TUKEY HSD POST-HOC: Number of Commands During Each Attack
              Multiple Comparison of Means - Tukey HSD, FWER=0.10              
       group1               group2        meandiff p-adj  lower   upper  reject
-------------------------------------------------------------------------------
   container-control container-treatment1  16.6677   0.0  9.3711 23.9642   True
   container-control container-treatment2  38.8152   0.0 31.7892 45.8411   True
   container-control container-treatment3  60.6591   0.0 53.4458 67.8724   True
container-treatment1 container-treatment2  22.1475   0.0 14.2368 30.0582   True
container-treatment1 container-treatment3  43.9914   0.0 35.9138  52.069   True


***
# Additional Tests/Graphs
If your group would like to conduct additional statistical tests and/or generate more graphs, you are welcome to do so. Please note that you are responsible for producing accurate results. Any graphs and/or analysis results that are deemed incorrect may result in a penalty. You are welcome to add additional markdown/code blocks!